# Behavior Label Generation


This notebook generates final behavior labels by combining clustering, anomaly detection, and peak demand results.

## Behavior Labels:
- **Normal Weekday Demand**: Regular weekday consumption patterns
- **Peak Demand Day**: High demand days (90th percentile)
- **Holiday or Low Demand Day**: Weekends or low consumption days
- **Abnormal Demand Day**: Anomalous patterns that aren't normal peaks

## Risk Levels:
- **Normal**: Normal Weekday Demand
- **Low**: Holiday or Low Demand Day  
- **Medium**: Peak Demand Day
- **High**: Abnormal Demand Day

## Steps:
1. Load daily profiles, clustering results, and anomaly results
2. Detect peak demand days using 90th percentile threshold
3. Combine all results by date
4. Generate behavior labels using rule-based logic
5. Create behavior reasons and risk levels
6. Save final behavior labels and summary


In [ ]:

import pandas as pd
import numpy as np
import json
from pathlib import Path
import sys

# Add project root to Python path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Set up paths
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
DAILY_PROFILES_PATH = DATA_PROCESSED_DIR / "daily_profiles.csv"
CLUSTERING_RESULTS_PATH = DATA_PROCESSED_DIR / "clustering_results.csv"
ANOMALY_RESULTS_PATH = DATA_PROCESSED_DIR / "anomaly_results.csv"
PEAK_DEMAND_RESULTS_PATH = DATA_PROCESSED_DIR / "peak_demand_results.csv"
BEHAVIOR_LABELS_PATH = DATA_PROCESSED_DIR / "behavior_labels.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Daily profiles path: {DAILY_PROFILES_PATH}")
print(f"Behavior labels path: {BEHAVIOR_LABELS_PATH}")


In [ ]:

# Load daily profiles
if not DAILY_PROFILES_PATH.exists():
    raise FileNotFoundError(f"Daily profiles not found at {DAILY_PROFILES_PATH}")

daily_profiles = pd.read_csv(DAILY_PROFILES_PATH)
print(f"Daily profiles loaded: {daily_profiles.shape}")

# Load clustering results
if not CLUSTERING_RESULTS_PATH.exists():
    raise FileNotFoundError(f"Clustering results not found at {CLUSTERING_RESULTS_PATH}")

clustering_results = pd.read_csv(CLUSTERING_RESULTS_PATH)
print(f"Clustering results loaded: {clustering_results.shape}")

# Load anomaly results
if not ANOMALY_RESULTS_PATH.exists():
    raise FileNotFoundError(f"Anomaly results not found at {ANOMALY_RESULTS_PATH}")

anomaly_results = pd.read_csv(ANOMALY_RESULTS_PATH)
print(f"Anomaly results loaded: {anomaly_results.shape}")

print(f"\nData loaded successfully!")
print(f"Date range: {daily_profiles['date'].min()} to {daily_profiles['date'].max()}")


In [ ]:

# Detect peak demand days using 90th percentile threshold
print("Detecting peak demand days...")

# Calculate 90th percentile threshold
peak_threshold_90 = daily_profiles['daily_peak'].quantile(0.90)
peak_threshold_75 = daily_profiles['daily_peak'].quantile(0.75)

print(f"90th percentile threshold: {peak_threshold_90:.2f} kW")
print(f"75th percentile threshold: {peak_threshold_75:.2f} kW")

# Apply peak detection logic
cond_90 = daily_profiles['daily_peak'] >= peak_threshold_90
cond_anomaly = anomaly_results['final_anomaly_flag'] & (daily_profiles['daily_peak'] >= peak_threshold_75)

is_peak_demand_day = cond_90 | cond_anomaly

# Generate peak reasons
peak_reasons = []
for i in range(len(daily_profiles)):
    reasons = []
    if cond_90.iloc[i]:
        reasons.append(f"daily_peak >= 90th percentile ({peak_threshold_90:.1f} kW)")
    if cond_anomaly.iloc[i]:
        reasons.append(f"anomaly with daily_peak >= 75th percentile ({peak_threshold_75:.1f} kW)")
    peak_reasons.append("; ".join(reasons) if reasons else "not a peak day")

# Create peak demand results
peak_results = daily_profiles[['date', 'daily_peak', 'daily_mean']].copy()
peak_results['peak_threshold'] = peak_threshold_90
peak_results['is_peak_demand_day'] = is_peak_demand_day.values
peak_results['peak_reason'] = peak_reasons

# Save peak demand results
peak_results.to_csv(PEAK_DEMAND_RESULTS_PATH, index=False)

print(f"Peak demand results saved to: {PEAK_DEMAND_RESULTS_PATH}")
print(f"Peak days identified: {is_peak_demand_day.sum()}")
print(f"Peak percentage: {is_peak_demand_day.mean()*100:.2f}%")


In [ ]:

# Combine all results by date
print("Combining all results by date...")

# Start with daily profiles
behavior_data = daily_profiles[['date', 'daily_mean', 'daily_peak', 'daily_std', 
                              'day_of_week', 'month', 'is_weekend']].copy()

# Merge clustering results
behavior_data = behavior_data.merge(
    clustering_results[['date', 'kmeans_cluster', 'dbscan_cluster', 'dbscan_noise_flag']],
    on='date', how='left'
)

# Merge anomaly results
behavior_data = behavior_data.merge(
    anomaly_results[['date', 'final_anomaly_flag', 'anomaly_reason']],
    on='date', how='left'
)

# Merge peak demand results
behavior_data = behavior_data.merge(
    peak_results[['date', 'is_peak_demand_day', 'peak_reason']],
    on='date', how='left'
)

print(f"Combined data shape: {behavior_data.shape}")
print(f"\nCombined data columns: {list(behavior_data.columns)}")


In [ ]:

# Calculate low demand threshold
low_threshold = behavior_data['daily_mean'].quantile(0.25)
print(f"Low demand threshold (25th percentile): {low_threshold:.2f} kW")

# Generate behavior labels
print("Generating behavior labels...")

labels = []
reasons = []
risk_levels = []

for _, row in behavior_data.iterrows():
    is_anomaly = bool(row['final_anomaly_flag'])
    is_peak = bool(row['is_peak_demand_day'])
    is_weekend = bool(row['is_weekend'])
    is_low = row['daily_mean'] < low_threshold
    
    if is_anomaly and not is_peak:
        label = "Abnormal Demand Day"
        risk = "High"
        reason = f"Flagged as anomaly ({row['anomaly_reason']}) but not a peak day"
    elif is_peak:
        label = "Peak Demand Day"
        risk = "Medium"
        reason = f"Peak day triggered: {row['peak_reason']}"
    elif is_weekend or is_low:
        label = "Holiday / Low Demand Day"
        risk = "Low"
        reason = f"Weekend or low demand (mean: {row['daily_mean']:.1f} kW < threshold: {low_threshold:.1f} kW)"
    else:
        label = "Normal Weekday Demand"
        risk = "Normal"
        reason = f"Normal demand pattern (mean: {row['daily_mean']:.1f} kW)"
    
    labels.append(label)
    reasons.append(reason)
    risk_levels.append(risk)

# Add labels to dataframe
behavior_data['behavior_label'] = labels
behavior_data['behavior_reason'] = reasons
behavior_data['behavior_risk_level'] = risk_levels

print("Behavior labels generated successfully!")


In [ ]:

# Calculate label distribution
total_days = len(behavior_data)
label_counts = behavior_data['behavior_label'].value_counts().to_dict()
label_percentages = {k: round(v / total_days * 100, 2) for k, v in label_counts.items()}

print("=== BEHAVIOR LABEL DISTRIBUTION ===")
for label, count in label_counts.items():
    print(f"{label}: {count} days ({label_percentages[label]}%)")

# Calculate risk distribution
risk_counts = behavior_data['behavior_risk_level'].value_counts().to_dict()
risk_percentages = {k: round(v / total_days * 100, 2) for k, v in risk_counts.items()}

print("\n=== RISK LEVEL DISTRIBUTION ===")
for risk, count in risk_counts.items():
    print(f"{risk}: {count} days ({risk_percentages[risk]}%)")

# Save behavior labels
behavior_data.to_csv(BEHAVIOR_LABELS_PATH, index=False)

print(f"\nBehavior labels saved to: {BEHAVIOR_LABELS_PATH}")
print(f"Final shape: {behavior_data.shape}")


In [ ]:

# Create and save behavior summary
behavior_summary = {
    'total_days': total_days,
    'label_counts': label_counts,
    'label_percentages': label_percentages,
    'risk_distribution': risk_counts,
    'risk_percentages': risk_percentages,
    'low_demand_threshold': round(float(low_threshold), 2),
    'peak_demand_threshold_90': round(float(peak_threshold_90), 2),
    'peak_demand_threshold_75': round(float(peak_threshold_75), 2),
    'peak_days_count': int(is_peak_demand_day.sum()),
    'peak_days_percentage': round(float(is_peak_demand_day.mean()) * 100, 2),
    'anomaly_days_count': int(behavior_data['final_anomaly_flag'].sum()),
    'anomaly_days_percentage': round(float(behavior_data['final_anomaly_flag'].mean()) * 100, 2),
    'data_info': {
        'date_range_start': behavior_data['date'].min(),
        'date_range_end': behavior_data['date'].max(),
        'weekend_days': int(behavior_data['is_weekend'].sum()),
        'weekday_days': int((~behavior_data['is_weekend']).sum())
    }
}

# Save summary
with open(MODELS_DIR / "behavior_label_summary.json", 'w') as f:
    json.dump(behavior_summary, f, indent=2)

print(f"Behavior summary saved to: {MODELS_DIR / 'behavior_label_summary.json'}")

print("\n=== FINAL BEHAVIOR LABELING SUMMARY ===")
print(f"Total days analyzed: {total_days}")
print(f"Date range: {behavior_data['date'].min()} to {behavior_data['date'].max()}")
print(f"Peak demand days: {behavior_summary['peak_days_count']} ({behavior_summary['peak_days_percentage']}%)")
print(f"Anomaly days: {behavior_summary['anomaly_days_count']} ({behavior_summary['anomaly_days_percentage']}%)")
print(f"\nSaved files:")
print(f"- {BEHAVIOR_LABELS_PATH}")
print(f"- {PEAK_DEMAND_RESULTS_PATH}")
print(f"- {MODELS_DIR / 'behavior_label_summary.json'}")
